In [1]:
DISEASE_YIELD_ADJUSTMENT = {
    "Corn_(maize)___healthy": 1.00,
    "Corn_(maize)___Common_rust_": 0.85,
    "Corn_(maize)___Northern_Leaf_Blight": 0.80,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 0.85,
}

def fuse_prediction(base_yield: float, predicted_class: str) -> dict:
    """
    Combines the tabular model's base yield prediction with a literature-cited
    disease severity adjustment. This is a rule-based fusion, not a jointly
    trained model — no dataset exists pairing real images with real yield
    outcomes, so the adjustment factor is sourced from agronomy literature
    (see Phase 8 documentation) rather than fitted from data.
    """
    factor = DISEASE_YIELD_ADJUSTMENT.get(predicted_class, 1.00)
    adjusted_yield = base_yield * factor

    return {
        "base_yield_tons_per_hectare": round(base_yield, 3),
        "predicted_disease": predicted_class,
        "yield_adjustment_factor": factor,
        "final_yield_tons_per_hectare": round(adjusted_yield, 3),
    }

# Example test
example = fuse_prediction(base_yield=6.2, predicted_class="Corn_(maize)___Northern_Leaf_Blight")
print(example)

{'base_yield_tons_per_hectare': 6.2, 'predicted_disease': 'Corn_(maize)___Northern_Leaf_Blight', 'yield_adjustment_factor': 0.8, 'final_yield_tons_per_hectare': 4.96}


In [6]:
import joblib
import tensorflow as tf
import numpy as np
import cv2

# Tabular model (linear regression pipeline, saved earlier)
tabular_model = joblib.load("../models/linear_regression_pipeline.pkl")  # adjust filename to whatever you actually saved it as

# CNN model (Stage 2 fine-tuned, from Kaggle)
cnn_model = tf.keras.models.load_model("../models/stage2_best.keras")

CLASS_NAMES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",  # index 0
    "Corn_(maize)___Common_rust_",                          # index 1
    "Corn_(maize)___Northern_Leaf_Blight",                  # index 2
    "Corn_(maize)___healthy",                               # index 3
]
# NOTE: this order must exactly match label_encoder.classes_ from the
# Kaggle notebook (alphabetical LabelEncoder output) — double check
# against class_mapping.json before trusting this

In [7]:
# Quick verification: run one known-labeled image through the model and
# check which output index corresponds to which class
import numpy as np

# Pick any one image path from your segmented dataset that you know the
# true label for (adjust path to wherever you have a sample image locally,
# or skip this if you don't have any local image files)
print(cnn_model.output_shape)  # should show (None, 4) — confirms 4-class output

(None, 4)


In [8]:
# NOTE: This ordering is based on Phase 1 documentation (sklearn LabelEncoder
# alphabetical sort with case-sensitive ASCII ordering: uppercase before
# lowercase). class_mapping.json (saved on Kaggle) was the authoritative
# source but is not available locally — treat this as a documented
# assumption, not a verified fact, until cross-checked against that file.
CLASS_NAMES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",  # index 0
    "Corn_(maize)___Common_rust_",                          # index 1
    "Corn_(maize)___Northern_Leaf_Blight",                  # index 2
    "Corn_(maize)___healthy",                               # index 3
]

In [10]:
import numpy as np
import cv2

CLASS_NAMES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
]

DISEASE_YIELD_ADJUSTMENT = {
    "Corn_(maize)___healthy": 1.00,
    "Corn_(maize)___Common_rust_": 0.85,
    "Corn_(maize)___Northern_Leaf_Blight": 0.80,
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 0.85,
}

IMG_SIZE = (224, 224)

def preprocess_image(image_path):
    """Mirrors Kaggle Cell 13's load_and_preprocess, for a local file path."""
    from tensorflow.keras.applications.efficientnet import preprocess_input
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)
    img = preprocess_input(img.astype(np.float32))
    return np.expand_dims(img, axis=0)  # add batch dimension

def predict_disease(image_path):
    img_batch = preprocess_image(image_path)
    probs = cnn_model.predict(img_batch, verbose=0)[0]
    predicted_idx = np.argmax(probs)
    return CLASS_NAMES[predicted_idx], float(probs[predicted_idx])

def predict_base_yield(tabular_features: dict):
    """tabular_features must match the exact column names/order from X_encoded."""
    X_input = pd.DataFrame([tabular_features])
    return float(tabular_model.predict(X_input)[0])

def fuse_prediction(image_path, tabular_features: dict):
    predicted_class, confidence = predict_disease(image_path)
    base_yield = predict_base_yield(tabular_features)
    factor = DISEASE_YIELD_ADJUSTMENT[predicted_class]
    final_yield = base_yield * factor

    return {
        "predicted_disease": predicted_class,
        "disease_confidence": round(confidence, 4),
        "base_yield_tons_per_hectare": round(base_yield, 3),
        "yield_adjustment_factor": factor,
        "final_yield_tons_per_hectare": round(final_yield, 3),
    }

In [15]:
test_image_path = r"C:\Users\rt\OneDrive\Desktop\008d9af0-7568-4a67-bb1a-0e915836ddc0___RS_NLB 4165 copy 2.jpg"

test_tabular_input = {
    "Rainfall_mm": 750.0,
    "Temperature_Celsius": 28.0,
    "Fertilizer_Used": 1,
    "Irrigation_Used": 1,
    "Days_to_Harvest": 100,
    "Region_East": 0, "Region_North": 0, "Region_South": 1, "Region_West": 0,
    "Soil_Type_Chalky": 0, "Soil_Type_Clay": 1, "Soil_Type_Loam": 0,
    "Soil_Type_Peaty": 0, "Soil_Type_Sandy": 0, "Soil_Type_Silt": 0,
    "Weather_Condition_Cloudy": 0, "Weather_Condition_Rainy": 1, "Weather_Condition_Sunny": 0,
}

result = fuse_prediction(test_image_path, test_tabular_input)
print(result)

{'predicted_disease': 'Corn_(maize)___Northern_Leaf_Blight', 'disease_confidence': 0.9998, 'base_yield_tons_per_hectare': 7.007, 'yield_adjustment_factor': 0.8, 'final_yield_tons_per_hectare': 5.605}


In [16]:
def fuse_prediction(image_path, tabular_features: dict):
    predicted_class, confidence = predict_disease(image_path)
    all_probs = predict_disease_probs(image_path)  # need all 4, not just the winner

    base_yield = predict_base_yield(tabular_features)

    p_healthy = all_probs[CLASS_NAMES.index("Corn_(maize)___healthy")]
    severity_scale = 1 - p_healthy  # 0 = model thinks fully healthy, 1 = model thinks fully diseased

    # Literature-based factor for a FULLY CONFIDENT disease case (our earlier
    # conservative estimates). At severity_scale=0, no reduction is applied
    # regardless of which disease "won" the argmax, since P(healthy) being
    # high means the model itself is uncertain the leaf is truly diseased.
    max_reduction = 1 - DISEASE_YIELD_ADJUSTMENT[predicted_class]  # e.g. 0.20 for NLB
    actual_reduction = max_reduction * severity_scale
    factor = 1 - actual_reduction

    final_yield = base_yield * factor

    return {
        "predicted_disease": predicted_class,
        "disease_confidence": round(confidence, 4),
        "p_healthy": round(float(p_healthy), 4),
        "severity_scale": round(float(severity_scale), 4),
        "base_yield_tons_per_hectare": round(base_yield, 3),
        "yield_adjustment_factor": round(float(factor), 4),
        "final_yield_tons_per_hectare": round(final_yield, 3),
    }

In [17]:
def predict_disease_probs(image_path):
    img_batch = preprocess_image(image_path)
    return cnn_model.predict(img_batch, verbose=0)[0]  # returns array of 4 probs

def predict_disease(image_path):
    probs = predict_disease_probs(image_path)
    predicted_idx = np.argmax(probs)
    return CLASS_NAMES[predicted_idx], float(probs[predicted_idx])

In [18]:
result = fuse_prediction(test_image_path, test_tabular_input)
print(result)


{'predicted_disease': 'Corn_(maize)___Northern_Leaf_Blight', 'disease_confidence': 0.9998, 'p_healthy': 0.0, 'severity_scale': 1.0, 'base_yield_tons_per_hectare': 7.007, 'yield_adjustment_factor': 0.8, 'final_yield_tons_per_hectare': np.float32(5.605)}


In [21]:
%pip install fastapi uvicorn python-multipart


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
with open("C:\Users\rt\Desktop\maize-yield-project\app\main.py") as f:
    print(f.read())

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2383901609.py, line 1)

In [5]:
import os
print(os.listdir(r"C:\Users\rt\Desktop\maize-yield-project\app"))

['main.py', 'model_utils.py', 'schemas.py', '__pycache__']
